### 2.1 卷积和池化层
1、输出通道数等于卷积核个数16；
输出高=输出宽=⌊(32 + 2*2 - 5)/2⌋ + 1 = 16；
最终特征图尺寸：16×16×16(通道×高×宽)

2、输入通道3，卷积核尺寸5×5；
单点乘法次数 = 3*5*5 = 75次

In [1]:
import numpy as np

def max_pool2d(x, kernel_size, stride=1, padding=0):
    # 填充
    x_pad = np.pad(x, ((0,0), (padding,padding), (padding,padding)), mode='constant')
    N, H, W = x_pad.shape
    kh, kw = kernel_size
    
    # 输出尺寸
    out_h = (H - kh) // stride + 1
    out_w = (W - kw) // stride + 1
    out = np.zeros((N, out_h, out_w))
    
    # 池化计算
    for i in range(out_h):
        for j in range(out_w):
            h_start = i * stride
            h_end = h_start + kh
            w_start = j * stride
            w_end = w_start + kw
            out[:, i, j] = np.max(x_pad[:, h_start:h_end, w_start:w_end], axis=(1,2))
    return out

# 测试
if __name__ == "__main__":
    test_x = np.random.rand(3, 32, 32)
    res = max_pool2d(test_x, kernel_size=(2,2), stride=2, padding=0)
    print("池化输出形状:", res.shape)

池化输出形状: (3, 16, 16)


### 3.1 VGG卷积参数量计算
#### 1. 单个5×5卷积层（无偏置）
公式：参数量 = 输入通道 × 卷积核高 × 卷积核宽 × 输出通道
#### 输入通道：$C$，输出通道：$C$，卷积核：$5\times5$
#### 参数数量 = C × 5 × 5 × C = 25*C²

#### 2. 两层串联3×3卷积层（无偏置，通道均为$C$）
两层串联3×3卷积，每层输入输出通道都是C，无偏置
#### 单层参数：C × 3 × 3 × C = 9*C²
#### 总参数 = 9*C² + 9*C² = 18*C²

#### 补充结论
同等输入输出通道下：$18C^2 < 25C^2$，两层$3\times3$堆叠参数量更少，且感受野等效$5\times5$，为VGG核心设计思想。

In [2]:
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 1),
            nn.ReLU()
        )
    def forward(self, x):
        return self.net(x)

# 测试
if __name__ == "__main__":
    nin_block = NiNBlock(3, 16, kernel_size=5, stride=1, padding=2)
    print(nin_block)

NiNBlock(
  (net): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU()
  )
)


### 4.1 BN批量归一化计算
#### BN公式

已知：$x_1=2,x_2=4,x_3=6,x_4=8,\gamma=2,\beta=1,\varepsilon=0$

##### 步骤1：求均值$\mu$
\[
\mu=\frac{2+4+6+8}{4}=5
\]
##### 步骤2：求方差$\sigma^2$
\[
\begin{align*}
\sigma^2&=\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}\\
&=\frac{9+1+1+9}{4}=5
\end{align*}
\]
##### 步骤3：逐个计算输出$y$
\[
\begin{align*}
y_1&=2\cdot\frac{2-5}{\sqrt5}+1=1-\frac{6}{\sqrt5}\\
y_2&=2\cdot\frac{4-5}{\sqrt5}+1=1-\frac{2}{\sqrt5}\\
y_3&=2\cdot\frac{6-5}{\sqrt5}+1=1+\frac{2}{\sqrt5}\\
y_4&=2\cdot\frac{8-5}{\sqrt5}+1=1+\frac{6}{\sqrt5}
\end{align*}
\]
#### 小数近似值
$y_1\approx-1.6833,\ y_2\approx0.1056,\ y_3\approx1.8944,\ y_4\approx3.6833$

In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        
        # 1×1卷积调整通道与尺寸
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, 1, stride)
        else:
            self.conv3 = None
    
    def forward(self, x):
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3:
            x = self.conv3(x)
        y += x
        return self.relu(y)

# 测试
if __name__ == "__main__":
    res_block = Residual(3, 64, use_1x1conv=True, stride=2)
    print(res_block)

Residual(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (conv3): Conv2d(3, 64, kernel_size=(1, 1), stride=(2, 2))
)


### 5.1 
#### 问题1
1. **底层特征提取层**：在ImageNet等大数据集预训练完成，已经学习到通用基础视觉特征（边缘、纹理、轮廓），这类特征具备通用性，适配多数图像任务。
    - 冻结/小学习率：避免大幅改动成熟参数，破坏已学习的通用特征；防止小目标数据集将底层参数带偏、引发过拟合。
2. **顶层输出层**：全新随机初始化，没有适配目标任务的参数，需要快速学习目标数据集专属分类/映射规则。
    - 大学习率：加快参数收敛，快速拟合新任务的数据分布。

#### 问题2（数据集小+源域相似，防过拟合策略）
1. **策略1：冻结全部特征提取主干网络**，仅训练最后全新的输出分类层，主干完全沿用预训练通用特征，最大限度保留源域有效知识。
2. **策略2：搭配图像增广**（随机裁剪、翻转、缩放、色彩扰动）扩充有限训练样本。
3. **策略3：输出层加入正则化**（Dropout、L2正则）约束顶层权重，降低过拟合风险。
4. **补充：全程使用更小学习率，禁止微调主干参数**。

In [4]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

# 测试
if __name__ == "__main__":
    print("图像增广管道：")
    print(transform)

图像增广管道：
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


### 6.1 IoU交并比计算
#### 已知参数
真实框 $A=[x_{a1},y_{a1},x_{a2},y_{a2}]=[10,10,50,50]$
预测框 $B=[x_{b1},y_{b1},x_{b2},y_{b2}]=[30,30,70,70]$

#### 1. 计算交集区域坐标
交集左上角：$x_1=\max(10,30)=30,\ y_1=\max(10,30)=30$
交集右下角：$x_2=\min(50,70)=50,\ y_2=\min(50,70)=50$
交集宽高：$w=50-30=20,\ h=50-30=20$
交集面积：$S_{inter}=20\times20=400$

#### 2. 分别计算两个框面积
$S_A=(50-10)\times(50-10)=40\times40=1600$
$S_B=(70-30)\times(70-30)=40\times40=1600$

#### 3. 并集面积与IoU
$$
S_{union}=S_A+S_B-S_{inter}=1600+1600-400=2800
$$
$$
IoU=\frac{S_{inter}}{S_{union}}=\frac{400}{2800}=\boldsymbol{\frac{1}{7}\approx0.1429}
$$

In [5]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(pred, label, epsilon=0.1):
    K = pred.size(-1)
    # 平滑标签
    smooth_label = torch.full_like(pred, epsilon/(K-1))
    smooth_label.scatter_(1, label.unsqueeze(1), 1-epsilon)
    # 交叉熵
    log_pred = F.log_softmax(pred, dim=-1)
    loss = -torch.sum(smooth_label * log_pred, dim=-1).mean()
    return loss

# 测试
if __name__ == "__main__":
    pred = torch.randn(4, 10)
    label = torch.tensor([0,1,2,3])
    loss = label_smoothing_cross_entropy(pred, label, epsilon=0.1)
    print("标签平滑交叉熵损失:", loss.item())

标签平滑交叉熵损失: 2.8266468048095703
